# Production upload helper

Set `project_name` and `profile_id` below, then run the cells in order (Python 3.10+). Leave `code_version = None` to read `Main_version/VERSION`, or set a version string to override it.

This notebook archives **everything inside `Production/Main_version/`**, including hidden files and empty directories, without a `Main_version/` wrapper. It writes an AES-256 password-protected ZIP to `Production/ALter_versions/<project_name>_<code_version>_<profile_id>.zip`. Reusing the same inputs replaces that local archive.

The final cell creates the Hugging Face Space `<account>/<project_name>` if needed, then replaces **all files on its `main` branch** with the new ZIP in one commit. Existing history and other branches remain. New Spaces are private by default and use the static SDK for archive storage; no running app is configured.

Credentials come only from `Production/.env`; their values are never printed or saved in this notebook. The `.env` outside `Main_version/` is not archived.


In [ ]:
%pip install --quiet "huggingface_hub>=1.0,<2" "pyzipper>=0.4,<1" "python-dotenv>=1,<2"


## Configuration

Use a Hugging Face repository name without an account prefix for `project_name`. Set `namespace` only to target an organization or a different account. Both tokens must have access to that Space, and the write token must permit repository creation when needed.

Add these keys to `Production/.env` for the selected profile, using your real values locally:

```dotenv
zip_password=YOUR_ZIP_PASSWORD
huggingface_1_read=YOUR_READ_TOKEN
huggingface_1_write=YOUR_WRITE_TOKEN
```

For `profile_id = 2`, the token keys become `huggingface_2_read` and `huggingface_2_write`.


In [ ]:
project_name = ""  # Required, for example: "my-project"
code_version = None  # None reads Main_version/VERSION; or set a version string.
profile_id = 1     # Integer or digit string, for example: 1 or "001"

namespace = None       # None uses the account owning the selected write token.
private_space = True   # Applies only when creating a new Space.
production_dir = None  # Auto-detected; alternatively set an absolute Production path.


## Load and validate settings

Automatic path detection works when the kernel starts in the repository, `Production/`, or a directory beneath it. Missing credentials or invalid inputs stop the workflow before any upload.


In [ ]:
from pathlib import Path
import re
import tempfile

import pyzipper
from dotenv import dotenv_values
from huggingface_hub import CommitOperationAdd, CommitOperationDelete, HfApi
from huggingface_hub.utils import validate_repo_id


def find_production_dir(override=None):
    if override is not None:
        candidates = [Path(override).expanduser().resolve()]
    else:
        current = Path.cwd().resolve()
        candidates = [
            candidate
            for parent in (current, *current.parents)
            for candidate in (parent, parent / "Production")
        ]
    for candidate in candidates:
        if (candidate / "Main_version").is_dir() and (candidate / ".env").is_file():
            return candidate
    raise FileNotFoundError(
        "Cannot find Main_version/ and .env together. "
        "Set production_dir to the absolute Production directory."
    )


def load_upload_settings(project_name, code_version, profile_id, production_dir=None):
    if not isinstance(project_name, str) or not project_name or "/" in project_name:
        raise ValueError("Set project_name to a Hugging Face repository name without '/'.")
    validate_repo_id(project_name)
    root = find_production_dir(production_dir)
    if code_version is None:
        version_path = root / "Main_version" / "VERSION"
        try:
            code_version = version_path.read_text(encoding="utf-8").strip()
        except FileNotFoundError as error:
            raise FileNotFoundError(f"Cannot read code version: {version_path}") from error
        if not re.fullmatch(r"[1-9][0-9]*", code_version):
            raise ValueError(f"{version_path} must contain one positive integer.")
    if not isinstance(code_version, str) or not re.fullmatch(
        r"[A-Za-z0-9][A-Za-z0-9._-]{0,63}", code_version
    ):
        raise ValueError(
            "Set code_version using 1-64 letters, digits, dots, underscores, or hyphens."
        )
    if isinstance(profile_id, bool) or not isinstance(profile_id, (int, str)):
        raise ValueError("profile_id must be an integer or a string of digits.")
    profile = str(profile_id)
    if not re.fullmatch(r"[0-9]+", profile):
        raise ValueError("profile_id must contain only digits.")

    values = dotenv_values(root / ".env", encoding="utf-8", interpolate=False)
    required = ("zip_password", f"huggingface_{profile}_read", f"huggingface_{profile}_write")
    missing = [key for key in required if not values.get(key) or not values[key].strip()]
    if missing:
        raise ValueError(f"Missing or empty keys in {root / '.env'}: {', '.join(missing)}")
    return {
        "root": root,
        "project_name": project_name,
        "archive_name": f"{project_name}_{code_version}_{profile}.zip",
        "zip_password": values["zip_password"].encode("utf-8"),
        "read_token": values[f"huggingface_{profile}_read"].strip(),
        "write_token": values[f"huggingface_{profile}_write"].strip(),
    }


## Build and verify the encrypted ZIP

All file contents use AES-256 encryption. The archive is read back with the password and checked before replacing any previous local ZIP. Symbolic links and special files stop packaging rather than following paths outside the source directory. Use an AES-capable ZIP reader, such as 7-Zip or `pyzipper`, to extract the result.


In [ ]:
def build_encrypted_zip(settings):
    source = (settings["root"] / "Main_version").resolve()
    entries = sorted(source.rglob("*"))
    if not any(path.is_file() for path in entries):
        raise ValueError("Main_version contains no files to upload.")
    for path in entries:
        if path.is_symlink() or not (path.is_file() or path.is_dir()):
            raise ValueError(f"Unsupported link or special file: {path.relative_to(source)}")

    output_dir = settings["root"] / "ALter_versions"
    if output_dir.resolve() == source or source in output_dir.resolve().parents:
        raise ValueError("The archive directory must be outside Main_version.")
    output_dir.mkdir(parents=True, exist_ok=True)
    archive_path = output_dir / settings["archive_name"]
    with tempfile.NamedTemporaryFile(
        dir=output_dir, prefix=".upload-", suffix=".zip.tmp", delete=False
    ) as temporary:
        temporary_path = Path(temporary.name)

    try:
        with pyzipper.AESZipFile(
            str(temporary_path), "w",
            compression=pyzipper.ZIP_DEFLATED,
            allowZip64=True,
        ) as archive:
            archive.setpassword(settings["zip_password"])
            archive.setencryption(pyzipper.WZ_AES, nbits=256, force_wz_aes_version=2)
            for path in entries:
                archive.write(str(path), arcname=path.relative_to(source).as_posix())

        with pyzipper.AESZipFile(str(temporary_path), "r") as archive:
            archive.setpassword(settings["zip_password"])
            expected = {
                path.relative_to(source).as_posix() + ("/" if path.is_dir() else "")
                for path in entries
            }
            if set(archive.namelist()) != expected:
                raise RuntimeError("ZIP verification failed: archive contents do not match.")
            for info in archive.infolist():
                if not info.is_dir() and (
                    not (info.flag_bits & 1) or info.wz_aes_strength != 3
                ):
                    raise RuntimeError("ZIP verification failed: a file is not AES-256 encrypted.")
            bad_file = archive.testzip()
            if bad_file is not None:
                raise RuntimeError(f"ZIP verification failed for: {bad_file}")

        temporary_path.replace(archive_path)
    finally:
        temporary_path.unlink(missing_ok=True)

    print(f"Verified encrypted archive: {archive_path}")
    print(f"Archive size: {archive_path.stat().st_size:,} bytes")
    return archive_path


## Replace the Space contents

The read token lists the current files and verifies the result; the write token creates the Space and commits the replacement. A commit based on the inspected revision removes old files (including README and `.gitattributes`) and adds the ZIP together. An existing ZIP with the same name is overwritten. If `main` changes during upload, the commit fails so you can inspect it and rerun.


In [ ]:
def replace_space_contents(archive_path, settings, namespace=None, private_space=True):
    if not isinstance(private_space, bool):
        raise ValueError("private_space must be True or False.")
    if namespace is not None and (
        not isinstance(namespace, str) or not namespace or "/" in namespace
    ):
        raise ValueError("namespace must be one account or organization name, or None.")
    archive_path = Path(archive_path)
    if not archive_path.is_file() or archive_path.name != settings["archive_name"]:
        raise ValueError("The verified ZIP is missing or its filename does not match.")

    read_api = HfApi(endpoint="https://huggingface.co", token=settings["read_token"])
    write_api = HfApi(endpoint="https://huggingface.co", token=settings["write_token"])
    owner = write_api.whoami()["name"]
    read_api.whoami()  # Validate the read token before creating or changing a Space.
    repo_id = f"{namespace or owner}/{settings['project_name']}"
    validate_repo_id(repo_id)
    print(f"Target Space: https://huggingface.co/spaces/{repo_id}")

    write_api.create_repo(
        repo_id=repo_id,
        repo_type="space",
        space_sdk="static",
        private=private_space,
        exist_ok=True,
    )
    current = read_api.repo_info(repo_id=repo_id, repo_type="space", revision="main")
    if not current.sha:
        raise RuntimeError("Cannot determine the Space's main revision; no files were replaced.")
    existing_files = read_api.list_repo_files(
        repo_id=repo_id, repo_type="space", revision=current.sha
    )
    operations = [
        CommitOperationDelete(path_in_repo=path, is_folder=False)
        for path in existing_files
        if path != archive_path.name
    ]
    operations.append(
        CommitOperationAdd(path_in_repo=archive_path.name, path_or_fileobj=str(archive_path))
    )
    commit = write_api.create_commit(
        repo_id=repo_id,
        repo_type="space",
        revision="main",
        parent_commit=current.sha,
        operations=operations,
        commit_message=f"Replace Space contents with {archive_path.name}",
    )
    uploaded_files = read_api.list_repo_files(
        repo_id=repo_id, repo_type="space", revision=commit.oid
    )
    if set(uploaded_files) != {archive_path.name}:
        raise RuntimeError(
            "The commit completed, but the resulting file list is not exactly the new ZIP. "
            f"Inspect https://huggingface.co/spaces/{repo_id}/tree/{commit.oid}"
        )
    space_url = f"https://huggingface.co/spaces/{repo_id}"
    print(f"Uploaded and verified: {archive_path.name}")
    print(f"Space: {space_url}")
    print(f"Files: https://huggingface.co/spaces/{repo_id}/tree/main")
    print(f"Commit: {commit.commit_url}")
    return {
        "repo_id": repo_id,
        "space_url": space_url,
        "archive_path": str(archive_path),
        "commit_url": commit.commit_url,
    }


## Run the upload

Running this cell builds the archive and replaces the selected remote Space's files. No upload happens until this cell is executed. Keep notebook outputs clear when sharing it.


In [ ]:
def upload_main_version(
    project_name, code_version, profile_id,
    namespace=None, private_space=True, production_dir=None,
):
    settings = load_upload_settings(
        project_name, code_version, profile_id, production_dir=production_dir
    )
    archive_path = build_encrypted_zip(settings)
    return replace_space_contents(
        archive_path, settings, namespace=namespace, private_space=private_space
    )


upload_result = upload_main_version(
    project_name=project_name,
    code_version=code_version,
    profile_id=profile_id,
    namespace=namespace,
    private_space=private_space,
    production_dir=production_dir,
)


Reference: [Hugging Face Hub API](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api) and [pyzipper encryption](https://github.com/danifus/pyzipper).
